In [ ]:
import torch
import torch.nn as nn

N_POINTS = 100

class ContourModel2_DeepTCN(nn.Module):

    def __init__(self, n_freqs=4):

        super().__init__()

        n_cond = 2 * n_freqs

        self.block1 = nn.Sequential(
            nn.Conv1d(6,64,3,padding=1,dilation=1),
            nn.BatchNorm1d(64),
            nn.GELU()
        )

        self.block2 = nn.Sequential(
            nn.Conv1d(64,64,3,padding=2,dilation=2),
            nn.BatchNorm1d(64),
            nn.GELU()
        )

        self.block3 = nn.Sequential(
            nn.Conv1d(64,128,3,padding=4,dilation=4),
            nn.BatchNorm1d(128),
            nn.GELU()
        )

        self.block4 = nn.Sequential(
            nn.Conv1d(128,128,3,padding=8,dilation=8),
            nn.BatchNorm1d(128),
            nn.GELU()
        )

        self.block5 = nn.Sequential(
            nn.Conv1d(128,256,3,padding=16,dilation=16),
            nn.BatchNorm1d(256),
            nn.GELU()
        )

        self.block6 = nn.Sequential(
            nn.Conv1d(256,256,3,padding=32,dilation=32),
            nn.BatchNorm1d(256),
            nn.GELU()
        )

        self.pool = nn.AdaptiveAvgPool1d(1)

        self.head = nn.Sequential(

            nn.Linear(256 + n_cond,512),
            nn.GELU(),

            nn.Dropout(0.2),

            nn.Linear(512,256),
            nn.GELU(),

            nn.Linear(256,200)

        )

    def forward(
        self,
        prev_pts,
        future_pts,
        motion,
        n_enc
    ):

        x = torch.cat(
            [
                prev_pts,
                future_pts,
                motion
            ],
            dim=-1
        )

        x = x.permute(0,2,1)

        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.block5(x)
        x = self.block6(x)

        x = self.pool(x).squeeze(-1)

        x = torch.cat(
            [x,n_enc],
            dim=1
        )

        x = self.head(x)

        return x.view(-1, N_POINTS, 2)